# Cancelable MinusFace v2 — Modular Implementation
### Imports from `models/`, `training/`, `eval/`, `utils/` packages

This notebook is a clean, thin orchestrator. All logic lives in the Python modules.
Run top-to-bottom on Google Colab T4. Manual steps: **mount Drive** + **wandb.login()**.

In [ ]:
# Block 0 — Install and clone repo
!pip install ptwt wandb --quiet

# ============================================================
# BEFORE RUNNING: set your GitHub repo URL below
# Steps:
#   1. Push this project to GitHub (if not done already)
#   2. Replace the URL below with your actual repo URL
#   3. If repo is private, use a personal access token:
#      https://<TOKEN>@github.com/username/repo.git
# ============================================================
import os, sys

REPO_URL  = "https://github.com/YOUR_USERNAME/cancelable-minusface"
REPO_PATH = '/content/cancelable-minusface'

if not os.path.exists(REPO_PATH):
    !git clone {REPO_URL} {REPO_PATH}

if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

import torch
print(f'PyTorch : {torch.__version__}')
print(f'GPU     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device  : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Block 1 — Mount Google Drive (manual approval required)
from google.colab import drive
drive.mount('/content/drive')

CKPT_DIR = '/content/drive/MyDrive/cancelable_minusface_checkpoints'
import os; os.makedirs(CKPT_DIR, exist_ok=True)
print(f'Checkpoint directory: {CKPT_DIR}')

In [ ]:
# Block 2 — Import modules
import torch
import torchvision.models as models
import torch.nn as nn
import wandb
import warnings; warnings.filterwarnings('ignore')

from models import WaveletMapper, UNetGenerator, CancelableTransform
from data import download_vggface2, build_dataloaders
from training import run_stage1, run_stage2, build_mlp
from eval import run_cancelability, run_non_invertibility, run_cancellation_demo
from utils import (
    make_checkpoint_fn, load_latest_checkpoint, restore_stage1, restore_stage2,
    plot_pipeline, plot_stage1, plot_stage2,
    plot_cancelability, plot_non_invertibility, plot_cancellation_demo,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using: {device}')

In [ ]:
# Block 3 — Hyperparameters
CFG = {
    'dataset':            'VGGFace2-112x112',
    'encoder':            'HaarWavelet-2level',
    'proj_dim':           512,
    'user_key':           12345,
    'batch_size':         64,
    'num_workers':        4,
    'imgs_per_identity':  100,       # stratified cap — keeps all 8631 IDs, ~863K total samples
    's1_epochs':          15,
    's2_epochs':          8,         # 8 epochs: MLP converges faster than full generator training
    'alpha':              5.0,
    'beta':               1.0,
    'lr_gen':             5e-3,
    'lr_fr':              1e-2,
    'lr_fp':              1e-3,
    'optimizer_s1':       'SGD',
    'optimizer_s2':       'Adam',
    'scheduler':          'CosineAnnealingLR',
    'momentum':           0.9,
    'weight_decay':       1e-4,
}

USE_AMP = True   # mixed precision — ~1.5-2x speedup on T4 Tensor Cores

print('Config:', CFG)
print(f'AMP enabled: {USE_AMP}')

In [ ]:
# Block 4 — W&B init (manual step: wandb.login() if not authenticated)
# wandb.login()

run = wandb.init(
    project='cancelable-minusface',
    name='v2-modular',
    config=CFG,
    mode='online',  # set 'disabled' to skip logging
)
print(f'W&B run: {run.name}  url: {run.url}')

In [ ]:
# Block 5 — Instantiate core models
mapper    = WaveletMapper().to(device)
ct        = CancelableTransform(proj_dim=CFG['proj_dim'])
generator = UNetGenerator().to(device)

_t = torch.randn(2, 3, 112, 112, device=device)
assert mapper.decode(mapper.encode(_t)).shape == _t.shape
print(f'WaveletMapper OK  encode-> {mapper.encode(_t).shape}')
print(f'UNetGenerator OK  params: {sum(p.numel() for p in generator.parameters()):,}')
print(f'CancelableTransform OK  in_dim={ct.in_dim}  out_dim={ct.out_dim}')

In [ ]:
# Block 5b — torch.compile (PyTorch >= 2.0; skipped gracefully on older builds)
# 'reduce-overhead' captures CUDA graph on first call — best for T4 with fixed input shapes.
try:
    generator  = torch.compile(generator,  mode='reduce-overhead')
    recognizer = torch.compile(recognizer, mode='reduce-overhead')
    print('torch.compile applied to generator and recognizer')
except Exception as e:
    print(f'torch.compile skipped ({e})')

In [ ]:
# Block 6 — Download VGGFace2 and build DataLoaders
image_dir = download_vggface2()
print(f'Image root: {image_dir}')

train_loader, val_loader, num_classes = build_dataloaders(
    image_dir,
    batch_size=CFG['batch_size'],
    num_workers=CFG['num_workers'],
    imgs_per_identity=CFG['imgs_per_identity'],   # stratified cap: 100 imgs/identity
)
print(f'Identities : {num_classes:,}')
print(f'Train/Val  : {len(train_loader.dataset):,} / {len(val_loader.dataset):,}')

run.config.update({'num_classes': num_classes})

In [ ]:
# Block 6b — Training time estimate (two warmup passes; second used — first is slow due to JIT)
import time
from torch.cuda.amp import GradScaler as _GradScaler, autocast as _autocast
import torch.optim as _optim

_use_amp = device.type == 'cuda'
imgs_w, lbls_w = next(iter(train_loader))
imgs_w, lbls_w = imgs_w.to(device), lbls_w.to(device)

_recognizer_w = __import__('torchvision').models.resnet18(num_classes=num_classes).to(device)
_recognizer_w.conv1 = __import__('torch').nn.Conv2d(21, 64, 7, stride=2, padding=3, bias=False).to(device)

_opt_warm   = _optim.SGD(
    list(generator.parameters()) + list(_recognizer_w.parameters()), lr=1e-3
)
_scaler_w   = _GradScaler(enabled=_use_amp)
_crit_l1_w  = __import__('torch').nn.L1Loss()
_crit_ce_w  = __import__('torch').nn.CrossEntropyLoss()

warmup_sec = None
for _warm in range(2):
    _opt_warm.zero_grad()
    t0 = time.time()
    with _autocast(enabled=_use_amp):
        _xw   = mapper.encode(imgs_w)
        _xpw  = generator(_xw)
        _rw   = _xw - _xpw
        _loss = 5.0 * _crit_l1_w(_xpw, _xw) + _crit_ce_w(_recognizer_w(_rw), lbls_w)
    _scaler_w.scale(_loss).backward()
    _scaler_w.step(_opt_warm)
    _scaler_w.update()
    warmup_sec = time.time() - t0   # overwrite — second pass is post-JIT

del _recognizer_w, _opt_warm, _scaler_w, _crit_l1_w, _crit_ce_w

n_batches    = len(train_loader)
S1_EPOCHS    = CFG['s1_epochs']
S2_EPOCHS    = CFG['s2_epochs']

sec_per_ep_s1 = warmup_sec * n_batches
sec_per_ep_s2 = warmup_sec * n_batches * 0.45   # Stage 2 skips generator backward

total_s1  = sec_per_ep_s1 * S1_EPOCHS
total_s2  = sec_per_ep_s2 * S2_EPOCHS
total_all = total_s1 + total_s2

print(f'Warmup batch : {warmup_sec:.2f}s  |  batches/epoch: {n_batches}')
print()
print(f'  Stage 1: ~{sec_per_ep_s1/60:.1f} min/epoch × {S1_EPOCHS} epochs = ~{total_s1/3600:.1f} hrs')
print(f'  Stage 2: ~{sec_per_ep_s2/60:.1f} min/epoch × {S2_EPOCHS} epochs = ~{total_s2/3600:.1f} hrs')
print(f'  Total estimated training time : ~{total_all/3600:.1f} hrs')

In [ ]:
# Block 7 — Stage 1: Joint generator + residue recognizer
import torch.nn as nn
from torch.cuda.amp import GradScaler

recognizer = __import__('torchvision').models.resnet18(num_classes=num_classes).to(device)
recognizer.conv1 = nn.Conv2d(21, 64, 7, stride=2, padding=3, bias=False).to(device)

# torch.compile for recognizer (fp compiled separately in Block 10)
try:
    recognizer = torch.compile(recognizer, mode='reduce-overhead')
    print('torch.compile applied to recognizer')
except Exception as e:
    print(f'torch.compile skipped for recognizer ({e})')

_use_amp = device.type == 'cuda'
scaler_s1 = GradScaler(enabled=_use_amp)

s1_ckpt_fn = make_checkpoint_fn(CKPT_DIR, 'stage1')
s1_state   = load_latest_checkpoint(CKPT_DIR, 'stage1')
s1_start   = 0
s1_gloss, s1_floss, s1_acc = [], [], []

if s1_state is not None:
    s1_start  = restore_stage1(s1_state, generator, recognizer, scaler=scaler_s1)
    s1_gloss  = s1_state.get('gen_losses', [])
    s1_floss  = s1_state.get('fr_losses',  [])
    s1_acc    = s1_state.get('val_accs',   [])

print(f'Starting Stage 1 from epoch {s1_start + 1}')

new_gloss, new_floss, new_acc = run_stage1(
    generator, recognizer, mapper, train_loader, val_loader,
    num_classes=num_classes, device=device,
    epochs=CFG['s1_epochs'], alpha=CFG['alpha'], beta=CFG['beta'],
    lr_gen=CFG['lr_gen'], lr_fr=CFG['lr_fr'],
    checkpoint_fn=s1_ckpt_fn, wandb_run=run,
    start_epoch=s1_start, scaler=scaler_s1,
)

s1_gloss.extend(new_gloss)
s1_floss.extend(new_floss)
s1_acc.extend(new_acc)

In [ ]:
# Block 8 — Stage 1 plots
plot_stage1(s1_gloss, s1_floss, s1_acc, out_path='stage1.png')
run.log({'stage1_chart': wandb.Image('stage1.png')})

In [ ]:
# Block 9 — Convergence guard (must pass before Stage 2)
final_gen_loss = s1_gloss[-1]
print(f'Final Stage 1 gen loss: {final_gen_loss:.4f}')

if final_gen_loss >= 0.05:
    print(
        f'WARNING: Generator has NOT converged (gen_loss={final_gen_loss:.4f} >= 0.05).\n'
        f'Residue r = x - x\' may still contain visible face structure.\n'
        f'Non-invertibility test will likely FAIL.\n'
        f"Recommended: increase s1_epochs to {CFG['s1_epochs'] + 5} and re-run Stage 1."
    )

assert final_gen_loss < 0.05, (
    f'Stage 1 gen_loss ({final_gen_loss:.4f}) must be < 0.05 before Stage 2. '
    f"Increase s1_epochs and re-run."
)
print('Generator converged. Proceeding to Stage 2.')

In [ ]:
# Block 10 — Stage 2: MLP on cancelable templates
from torch.cuda.amp import GradScaler

fp = build_mlp(CFG['proj_dim'], num_classes).to(device)

try:
    fp = torch.compile(fp, mode='reduce-overhead')
    print('torch.compile applied to fp')
except Exception as e:
    print(f'torch.compile skipped for fp ({e})')

_use_amp = device.type == 'cuda'
scaler_s2 = GradScaler(enabled=_use_amp)

s2_ckpt_fn = make_checkpoint_fn(CKPT_DIR, 'stage2')
s2_state   = load_latest_checkpoint(CKPT_DIR, 'stage2')
s2_start   = 0
s2_loss, s2_acc = [], []

if s2_state is not None:
    s2_start = restore_stage2(s2_state, fp, scaler=scaler_s2)
    s2_loss  = s2_state.get('losses',   [])
    s2_acc   = s2_state.get('val_accs', [])

print(f'Starting Stage 2 from epoch {s2_start + 1}')

new_loss, new_acc = run_stage2(
    generator, fp, mapper, ct, train_loader, val_loader, device,
    user_key=CFG['user_key'], epochs=CFG['s2_epochs'], lr=CFG['lr_fp'],
    checkpoint_fn=s2_ckpt_fn, wandb_run=run,
    start_epoch=s2_start, scaler=scaler_s2,
)

s2_loss.extend(new_loss)
s2_acc.extend(new_acc)

In [ ]:
# Block 11 — Stage 2 plots
plot_stage2(s2_loss, s2_acc, out_path='stage2.png')
run.log({'stage2_chart': wandb.Image('stage2.png')})

In [ ]:
# Block 12 — Pipeline visualisation
import torch.nn.functional as F

imgs_v, _ = next(iter(val_loader))
img = imgs_v[0:1].to(device)
generator.eval()
with torch.no_grad():
    xv   = mapper.encode(img)
    xpv  = generator(xv)
    rv   = xv - xpv
    tmpl = ct.transform(rv, key=CFG['user_key'])
    Xp   = mapper.decode(xpv)
    R    = mapper.decode(rv)

plot_pipeline(img, Xp, R, tmpl, out_path='pipeline.png')
run.log({'pipeline': wandb.Image('pipeline.png')})

In [ ]:
# Block 13 — Cancelability experiment
KEY_A, KEY_B = 11111, 99999

canc_results = run_cancelability(
    generator, mapper, ct, val_loader, device,
    key_a=KEY_A, key_b=KEY_B, n_pairs=500, max_samples=1000,
)

run.log({
    'eval/genuine_sk_mean': float(canc_results['genuine_sk'].mean()),
    'eval/genuine_ck_mean': float(canc_results['genuine_ck'].mean()),
    'eval/impostor_mean':   float(canc_results['impostor'].mean()),
    'eval/auc_same_key':    canc_results['auc_same_key'],
    'eval/auc_cross_key':   canc_results['auc_cross_key'],
    'eval/unlinkable':      int(canc_results['unlinkable']),
})

In [ ]:
# Block 14 — Cancelability plots
plot_cancelability(
    canc_results['genuine_sk'], canc_results['genuine_ck'], canc_results['impostor'],
    canc_results['fpr_sk'], canc_results['tpr_sk'], canc_results['auc_same_key'],
    canc_results['fpr_ck'], canc_results['tpr_ck'], canc_results['auc_cross_key'],
    out_path='cancelability.png',
)
run.log({'cancelability': wandb.Image('cancelability.png')})

In [ ]:
# Block 15 — Non-invertibility experiment
ni_results = run_non_invertibility(
    generator, mapper, ct, val_loader, device, key=KEY_A,
)

run.log({
    'eval/recovery_sim_mean': ni_results['mean_sim'],
    'eval/non_invertible': int(ni_results['pass_test']),
})

In [ ]:
# Block 16 — Non-invertibility plot
plot_non_invertibility(
    ni_results['r_true'], ni_results['r_est'],
    ni_results['recovery_sim'], mapper,
    out_path='noninvert.png',
)

In [ ]:
# Block 17 — Cancellation demo
demo_results = run_cancellation_demo(
    generator, mapper, ct, val_loader, device,
    k_orig=55555, k_new=77777, auth_threshold=0.2,
)

run.log({
    'eval/sim_before':       demo_results['sim_before'],
    'eval/template_linkage': demo_results['template_linkage'],
    'eval/sim_after':        demo_results['sim_after'],
})

In [ ]:
# Block 18 — Cancellation demo plot
plot_cancellation_demo(demo_results, out_path='cancellation_demo.png')

In [ ]:
# Block 19 — Final summary table
import numpy as np

print('='*70)
print('CANCELABLE MINUSFACE v2 — FINAL RESULTS SUMMARY')
print('='*70)

rows = [
    ('Dataset',             f"VGGFace2  {num_classes:,} identities"),
    ('Encoder',             'Haar Wavelet 2-level  (B,3,112,112)->(B,21,56,56)'),
    ('Generator',           f"U-Net  {sum(p.numel() for p in generator.parameters()):,} params"),
    ('Cancelable T(r,K)',   f"Gaussian proj 65856->{CFG['proj_dim']}  (CPU only)"),
    ('Recognizer f_p',      f"MLP  {sum(p.numel() for p in fp.parameters()):,} params"),
    ('', ''),
    ('S1 final gen loss',   f"{s1_gloss[-1]:.4f}  (target: < 0.05)"),
    ('S1 peak val acc',     f"{max(s1_acc):.2f}%"),
    ('S2 peak val acc',     f"{max(s2_acc):.2f}%"),
    ('', ''),
    ('Genuine AUC (same-key)',  f"{canc_results['auc_same_key']:.4f}  (usability; higher = better)"),
    ('Cross-key AUC',           f"{canc_results['auc_cross_key']:.4f}  (unlinkability; 0.5 = perfect)"),
    ('Non-invertibility sim',   f"{ni_results['mean_sim']:.4f}  (lower = safer; pass: < 0.3)"),
    ('Template linkage',        f"{demo_results['template_linkage']:.4f}  (lower = more unlinkable)"),
    ('', ''),
    ('Unlinkability',   'PASS' if canc_results['unlinkable'] else 'PARTIAL (needs more training)'),
    ('Non-invertibility', 'PASS' if ni_results['pass_test'] else 'REVIEW'),
]

for k, v in rows:
    if k == '':
        print()
    else:
        print(f'  {k:<32} {v}')

print('='*70)
print('Output files: pipeline.png  stage1.png  stage2.png')
print('              cancelability.png  noninvert.png  cancellation_demo.png')
print('='*70)

# Accuracy delta: how much accuracy is traded for cancelability?
_s1_peak   = max(s1_acc)
_s2_peak   = max(s2_acc)
delta      = _s1_peak - _s2_peak
_target_ok = abs(delta) <= 2.0
_delta_label = '✓ within 1-2% target' if _target_ok else '✗ exceeds 1-2% target — review projection dim'
print()
print(f'  {"Stage 1 peak acc (raw residue)":<32} {_s1_peak:.2f}%')
print(f'  {"Stage 2 peak acc (cancelable)":<32} {_s2_peak:.2f}%')
print(f'  {"Accuracy delta":<32} {abs(delta):.2f}%  {_delta_label}')
print()

run.log({'accuracy_delta': delta})
run.summary.update({
    's1_final_gen_loss':  s1_gloss[-1],
    's1_peak_acc':        _s1_peak,
    's2_peak_acc':        _s2_peak,
    'auc_same_key':       canc_results['auc_same_key'],
    'auc_cross_key':      canc_results['auc_cross_key'],
    'recovery_sim_mean':  ni_results['mean_sim'],
    'template_linkage':   demo_results['template_linkage'],
    'unlinkable':         int(canc_results['unlinkable']),
    'non_invertible':     int(ni_results['pass_test']),
    'accuracy_delta':     delta,
})
run.finish()
print('W&B run finished.')